# RSI Experimental Framework — Open-Weight Walkthrough

This notebook runs the repository's **actual** code on a fresh Colab runtime. It has two
clearly separated paths:

1. **Deterministic harness (default, CPU-safe).** Reproduces the committed
   `HARNESS_BASELINE_NOT_LLM` artifacts and exercises the candidate-validation gate.
   This is harness evidence, not a model experiment.
2. **Optional local open-weight provider (`--provider llm`).** Runs the real HuggingFace
   Transformers path when an accelerator is available. No paid APIs, no private tokens,
   no gated models.

The notebook never overwrites the committed `results/` artifacts: every run writes to a
temporary directory.

**Scientific boundary:** no real-model RSI experiment is recorded in this repository. A run
here is a local demonstration on a toy dataset — not research evidence and not a
significance result.


## 1. Runtime check

In [ ]:
import platform
import sys

print("python:", sys.version.split()[0])
print("platform:", platform.platform())

try:
    import torch
    print("torch:", torch.__version__)
    print("cuda available:", torch.cuda.is_available())
    if torch.cuda.is_available():
        print("gpu:", torch.cuda.get_device_name(0))
except ImportError:
    print("torch: not installed (only needed for the optional --provider llm path)")


## 2. Clone and import the repository

This clones `main` and imports the real `rsi_framework` package — nothing is reimplemented
here. Re-running the cell is safe: the clone is skipped if it already exists.


In [ ]:
import importlib
import os
import subprocess
import sys

REPO_DIR = "rsi-experimental-framework"

if not os.path.isdir(REPO_DIR):
    subprocess.check_call([
        "git", "clone", "--depth", "1",
        "https://github.com/rustfuture/rsi-experimental-framework.git",
        REPO_DIR,
    ])

os.chdir(REPO_DIR)
sys.path.insert(0, os.getcwd())
importlib.invalidate_caches()

import rsi_framework

print("working directory:", os.getcwd())
print("imported rsi_framework from:", os.path.dirname(rsi_framework.__file__))


## 3. Deterministic harness (CPU-safe default)

Runs the committed deterministic baseline (`config/default.json`) with the standard
library only and writes into a temporary directory.


In [ ]:
import json
import tempfile

run_dir = tempfile.mkdtemp(prefix="rsi_deterministic_")

subprocess.check_call([
    sys.executable, "-m", "rsi_framework",
    "--config", "config/default.json",
    "--output", run_dir,
], cwd=os.getcwd())

with open(os.path.join(run_dir, "first_run.json"), encoding="utf-8") as handle:
    first_run = json.load(handle)

summary = {
    "method_label": first_run["method_label"],
    "experiment_version": first_run["experiment_version"],
    "outcome": first_run["outcome"],
    "baseline_train_accuracy": first_run["baseline"]["train"]["accuracy"],
    "final_train_accuracy": first_run["final"]["train"]["accuracy"],
    "final_dev_accuracy": first_run["final"]["dev"]["accuracy"],
    "final_heldout_accuracy": first_run["final"]["heldout"]["accuracy"],
    "accepted_new_changes": first_run["accepted_new_change_count"],
    "rejected_proposals": first_run["rejected_proposal_count"],
}
print(json.dumps(summary, indent=2))
print("\nartifacts:", sorted(os.listdir(run_dir)))


## 4. Verify the committed artifacts (read-only)

Confirms that `results/report.md` and the README generated block match the committed
JSON artifacts. This is the same check CI runs; it does not modify anything.


In [ ]:
subprocess.check_call([
    sys.executable, "-m", "rsi_framework.reporting",
    "--results", "results",
    "--readme", "README.md",
    "--check",
], cwd=os.getcwd())


## 5. Candidate-validation gate

Every proposal from every provider is validated before it can be scored. The gate rejects
no-ops, multi-element edits, out-of-vocabulary keywords, keywords shared across polarities,
and out-of-bound bias.


In [ ]:
from rsi_framework.core import Candidate, Policy
from rsi_framework.providers import ProposalError, validate_proposal

current = Candidate(Policy(("good",), ("bad",), 0), 0, None, "baseline")

validate_proposal(current, Policy(("good", "clear"), ("bad",), 0))
print("accepted: single keyword edit within the vocabulary")

rejected = [
    ("multi-element edit", Policy(("good", "clear", "safe"), ("bad",), 0)),
    ("keyword in both polarities", Policy(("good",), ("good", "bad"), 0)),
    ("bias outside the bound", Policy(("good",), ("bad",), 9)),
]
for label, policy in rejected:
    try:
        validate_proposal(current, policy)
    except ProposalError as error:
        print(f"rejected ({label}): {error}")
    else:
        raise AssertionError(f"expected a rejection: {label}")


## 6. Optional: local open-weight provider (requires an accelerator)

Set `RUN_LLM_EXPERIMENT = True` on a GPU runtime. The default one-click path stays
deterministic, so this cell notebook runs on a CPU-only runtime.

Model choice: [`Qwen/Qwen2.5-0.5B-Instruct`](https://huggingface.co/Qwen/Qwen2.5-0.5B-Instruct)
is public, ungated, Apache-2.0, and small enough for a free GPU session. Change `MODEL_ID`
to any compatible causal-LM checkpoint; a private token is never required for the default.


In [ ]:
RUN_LLM_EXPERIMENT = False   # set to True on a GPU runtime
MODEL_ID = "Qwen/Qwen2.5-0.5B-Instruct"
DEVICE = "auto"              # auto -> cuda, then mps, then cpu

print("RUN_LLM_EXPERIMENT:", RUN_LLM_EXPERIMENT)
print("MODEL_ID:", MODEL_ID)

if RUN_LLM_EXPERIMENT:
    import importlib.util
    if importlib.util.find_spec("torch") is None:
        raise RuntimeError(
            "torch is not installed. Install it in the dependency cell and re-run from a "
            "clean runtime; the notebook will not silently fall back to the deterministic path."
        )
    import torch
    has_accelerator = torch.cuda.is_available() or (
        getattr(torch.backends, "mps", None) is not None and torch.backends.mps.is_available()
    )
    if not has_accelerator:
        raise RuntimeError(
            "RUN_LLM_EXPERIMENT=True but no CUDA/MPS accelerator is available. "
            "Switch this Colab runtime to GPU, or set RUN_LLM_EXPERIMENT=False."
        )
    print("accelerator detected; device request:", DEVICE)
else:
    print("Deterministic path selected. Set RUN_LLM_EXPERIMENT=True on a GPU runtime.")


## 7. Install the optional provider dependencies

Only runs when `RUN_LLM_EXPERIMENT` is `True`. The deterministic path needs none of these.


In [ ]:
if RUN_LLM_EXPERIMENT:
    subprocess.check_call([sys.executable, "-m", "pip", "install", "-q", "transformers"])
    print("installed transformers")
else:
    print("skipped: the deterministic path uses only the Python standard library")


## 8. Run `--provider llm` (optional)

Uses the real CLI end to end. The demo config reduces `generations` from the committed 8 to
2 so a small model stays within a free GPU session; outputs go to a temporary directory and
never touch the committed artifacts.

An unavailable runtime or requested device exits with `EXPERIMENT_BLOCKED_BY_RUNTIME`
rather than substituting another device.


In [ ]:
if RUN_LLM_EXPERIMENT:
    with open("config/default.json", encoding="utf-8") as handle:
        demo_config = json.load(handle)
    demo_config["generations"] = 2

    config_dir = tempfile.mkdtemp(prefix="rsi_llm_config_")
    demo_config_path = os.path.join(config_dir, "demo_config.json")
    with open(demo_config_path, "w", encoding="utf-8") as handle:
        json.dump(demo_config, handle, indent=2)

    llm_dir = tempfile.mkdtemp(prefix="rsi_llm_")
    subprocess.check_call([
        sys.executable, "-m", "rsi_framework",
        "--provider", "llm",
        "--model", MODEL_ID,
        "--device", DEVICE,
        "--config", demo_config_path,
        "--output", llm_dir,
    ], cwd=os.getcwd())

    with open(os.path.join(llm_dir, "first_run.json"), encoding="utf-8") as handle:
        llm_run = json.load(handle)
    print(json.dumps({
        "provider": "llm",
        "model": MODEL_ID,
        "outcome": llm_run["outcome"],
        "final_dev_accuracy": llm_run["final"]["dev"]["accuracy"],
        "final_heldout_accuracy": llm_run["final"]["heldout"]["accuracy"],
        "accepted_new_changes": llm_run["accepted_new_change_count"],
        "rejected_proposals": llm_run["rejected_proposal_count"],
        "note": "local demo only; not committed research evidence",
    }, indent=2))
else:
    print("RUN_LLM_EXPERIMENT is False — the deterministic artifacts above are the committed baseline.")


## 9. Locate or export artifacts

In [ ]:
if RUN_LLM_EXPERIMENT:
    print("llm demo artifacts:", sorted(os.listdir(llm_dir)))
else:
    print("deterministic artifacts:", run_dir)

# Export example (Colab only):
#   import shutil
#   archive = shutil.make_archive("/tmp/rsi_artifacts", "zip", run_dir)
#   from google.colab import files
#   files.download(archive)


## 10. How to read these results

- The deterministic run above reproduces the committed `HARNESS_BASELINE_NOT_LLM` record:
  a keyword-policy search over a fixed 32-sentence toy pool. It is harness evidence.
- The optional `--provider llm` run is a **local demonstration**. It is not committed,
  not a research result, and not a significance claim. The repository records no
  real-model RSI experiment.
- The held-out split is measured only after selection; it never participates in candidate
  generation, scoring, or selection.
